# Anchored Pilot DPO Replication

Objective: train one DPO QLoRA adapter each for ARG, SWE, and USA using the anchored 172-row samples, then evaluate every adapter on every held-out country split to test whether within-group reward recovery is stronger than between-group reward recovery.

Success criteria:
- three adapter directories are produced;
- a 3x3 adapter/eval reward-recovery matrix is produced;
- the run report states whether diagonal cells beat off-diagonal cells and flags data caveats.

## Manual prerequisites

Run this notebook in a Colab GPU runtime. A100 or L4 is preferred. Confirm Hugging Face access to `meta-llama/Llama-3.1-8B-Instruct`, then authenticate with a read token in the login cell below.

In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "DPO_anchored_pilot_experiment").exists():
    candidates = list(Path("/content").glob("*/DPO_anchored_pilot_experiment"))
    if candidates:
        REPO_ROOT = candidates[0].parent

EXPERIMENT_ROOT = REPO_ROOT / "DPO_anchored_pilot_experiment"
if not EXPERIMENT_ROOT.exists():
    raise FileNotFoundError(
        "Could not find DPO_anchored_pilot_experiment. Set REPO_ROOT to the cloned repo root."
    )

sys.path.insert(0, str(EXPERIMENT_ROOT))
print("REPO_ROOT:", REPO_ROOT)
print("EXPERIMENT_ROOT:", EXPERIMENT_ROOT)

In [ ]:
import subprocess

requirements = EXPERIMENT_ROOT / "requirements-colab.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])
print("Installed Colab dependencies from", requirements)

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## Configuration

The output directory is gitignored and can be redirected to Google Drive if desired. Keep the source JSONL files unchanged.

In [ ]:
from dpo_anchored.config import COUNTRIES, ExperimentConfig

OUTPUT_ROOT = EXPERIMENT_ROOT / "outputs" / "anchored_pilot_v1_dpo"
config = ExperimentConfig(repo_root=REPO_ROOT, output_root=OUTPUT_ROOT)
config.ensure_output_dirs()

print("Countries:", COUNTRIES)
print("Model:", config.model_name)
print("Output root:", config.output_root)

## Data validation and deterministic splits

This checks required fields, confirms the 172-row source files, reports dimension counts, and writes deterministic train/eval splits.

In [ ]:
from dpo_anchored.data import prepare_splits, print_reports, validate_sources, write_validation_report

reports = validate_sources(config)
print_reports(reports)
split_summary = prepare_splits(config)
report_path = write_validation_report(config, reports, split_summary=split_summary)
print("Split summary:", split_summary)
print("Validation report:", report_path)

## Runtime smoke check

This verifies CUDA, Hugging Face model access, tokenizer loading, 4-bit model loading, and one sample log probability before doing full work.

In [ ]:
from dpo_anchored.modeling import runtime_smoke_check

runtime_smoke_check(config)

## Precompute reference logprobs and train adapters

Run this cell country by country if GPU memory is tight. The default trains ARG, SWE, and USA sequentially.

In [ ]:
from dpo_anchored.modeling import precompute_reference_logps, train_adapter, write_training_metadata

RUN_COUNTRIES = list(COUNTRIES)  # edit to ["ARG"] etc. for one-country runs
write_training_metadata(config)

for country in RUN_COUNTRIES:
    print(f"=== {country}: precompute reference logprobs ===")
    precompute_reference_logps(config, country)
    print(f"=== {country}: train adapter ===")
    train_adapter(config, country)

## Evaluation smoke test

Evaluate one adapter on two examples per country without generated answers. Run this before the full 3x3 evaluation.

In [ ]:
from dpo_anchored.modeling import evaluate_adapter_on_all_countries

smoke_df = evaluate_adapter_on_all_countries(
    config,
    adapter_country="ARG",
    max_examples=2,
    generate_answers=False,
)
smoke_df.groupby(["adapter_country", "eval_country"]).size()

## Full 3x3 evaluation

This evaluates every adapter on every held-out country split. Generated answers are enabled by default for qualitative inspection.

In [ ]:
from dpo_anchored.modeling import run_full_evaluation

combined_file = run_full_evaluation(config, generate_answers=True)
print("Combined evaluation:", combined_file)

## Summaries and report

These tables are the main discussion artifacts: adapter summary, dimension summary, specialization matrices, own-vs-other summary, and a short Markdown report.

In [ ]:
from dpo_anchored.analysis import summarize_results, write_run_report

outputs = summarize_results(config)
report = write_run_report(config)
for name, output_path in outputs.items():
    print(f"{name}: {output_path}")
print("report:", report)

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

adapter_summary = pd.read_csv(config.results_dir / "reward_recovery_adapter_summary.csv")
mean_matrix = pd.read_csv(config.results_dir / "specialization_matrix_mean_reward_delta.csv", index_col=0)
accuracy_matrix = pd.read_csv(config.results_dir / "specialization_matrix_preference_accuracy.csv", index_col=0)
own_vs_other = pd.read_csv(config.results_dir / "own_vs_other_summary.csv")

display(adapter_summary)
display(mean_matrix)
display(accuracy_matrix)
display(own_vs_other)
display(Markdown((config.reports_dir / "run_report.md").read_text()))

## Notes for discussion

Use the run report and the two specialization matrices to discuss whether the new anchored synthetic pipeline produces learnable group-conditioned preference signals. Each eval split has 35 rows; treat dimension-level cells with very small `n` as descriptive only.